# AudialMind — Assignment 1
## Task 1 · RAVDESS Dataset Analysis

**Dataset:** Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS)  
**Covers:** Part A (Filename Decoding), Part B (Dataset Statistics), Part C (Written Reflection)

---
## 0. Environment Setup & Dataset Download
Run this cell first to install dependencies and download the RAVDESS dataset.

In [ ]:
# ── Install required libraries ──────────────────────────────────────────────
!pip install -q librosa soundfile tqdm kaggle

import os, zipfile, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ── matplotlib style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print('All imports successful.')

In [ ]:
# ── Option 1: Download via torchaudio (speech-only subset, easiest) ──────────
# This downloads only the speech recordings (1440 files).

import torchaudio

DATA_ROOT = './data/ravdess'
os.makedirs(DATA_ROOT, exist_ok=True)

# torchaudio will download & cache automatically
# NOTE: Set download=True on first run; set False afterwards to save time
dataset_torch = torchaudio.datasets.RAVDESS(
    root=DATA_ROOT,
    download=True
)
print(f'torchaudio RAVDESS: {len(dataset_torch)} files')

In [ ]:
# ── Option 2: Download from Kaggle (full dataset including song modality) ────
# Use this if you want the full dataset or if torchaudio fails.
#
# Steps:
#   1. Go to https://www.kaggle.com/account → Create New API Token → download kaggle.json
#   2. Upload kaggle.json to Colab, then run the cell below.

# from google.colab import files
# files.upload()          # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio
# !unzip -q ravdess-emotional-speech-audio.zip -d ./data/ravdess_kaggle

# ── Locate all .wav files under DATA_ROOT ────────────────────────────────────
# Adjust the glob pattern if you used Option 2 (path will differ)
WAV_FILES = sorted(glob.glob(os.path.join(DATA_ROOT, '**', '*.wav'), recursive=True))

# Filter to speech modality only (modality code = '03')
WAV_FILES = [f for f in WAV_FILES if os.path.basename(f).split('-')[0] == '03']

print(f'Found {len(WAV_FILES)} speech .wav files')
print('Example path:', WAV_FILES[0] if WAV_FILES else 'None found')

---
## Part A — Filename Decoding

RAVDESS filenames follow the pattern `MM-VC-EM-IN-ST-RE-AC.wav`  
where each segment encodes metadata as a two-digit code.

In [ ]:
# ── Lookup tables for all RAVDESS codes ─────────────────────────────────────

MODALITY_MAP = {
    '01': 'full-AV',
    '02': 'video-only',
    '03': 'audio-only'
}

VOCAL_CHANNEL_MAP = {
    '01': 'speech',
    '02': 'song'
}

EMOTION_MAP = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

INTENSITY_MAP = {
    '01': 'normal',
    '02': 'strong'
}

STATEMENT_MAP = {
    '01': 'Kids are talking by the door',
    '02': 'Dogs are sitting by the door'
}

REPETITION_MAP = {
    '01': '1st repetition',
    '02': '2nd repetition'
}


def parse_ravdess_filename(path: str) -> dict:
    """
    Parse a RAVDESS filename and return a dictionary with human-readable labels.

    Parameters
    ----------
    path : str
        Full path or basename of a RAVDESS .wav file.
        Expected format: MM-VC-EM-IN-ST-RE-AC.wav
        Example: 03-01-05-01-01-01-24.wav

    Returns
    -------
    dict
        Keys: modality, vocal_channel, emotion, intensity,
              statement, repetition, actor
        Values: human-readable strings

    Raises
    ------
    ValueError
        If the filename does not match the expected 7-segment pattern.
    """
    basename = os.path.basename(path)          # strip directory
    stem     = os.path.splitext(basename)[0]   # remove .wav
    parts    = stem.split('-')

    if len(parts) != 7:
        raise ValueError(
            f"Expected 7 dash-separated segments, got {len(parts)} in '{basename}'"
        )

    modality_code, vocal_code, emotion_code, \
    intensity_code, statement_code, repetition_code, actor_code = parts

    return {
        'modality':     MODALITY_MAP.get(modality_code,     f'unknown ({modality_code})'),
        'vocal_channel': VOCAL_CHANNEL_MAP.get(vocal_code,  f'unknown ({vocal_code})'),
        'emotion':      EMOTION_MAP.get(emotion_code,       f'unknown ({emotion_code})'),
        'intensity':    INTENSITY_MAP.get(intensity_code,   f'unknown ({intensity_code})'),
        'statement':    STATEMENT_MAP.get(statement_code,   f'unknown ({statement_code})'),
        'repetition':   REPETITION_MAP.get(repetition_code, f'unknown ({repetition_code})'),
        'actor':        f'Actor {int(actor_code):02d}'
    }


# ── Quick smoke-test ─────────────────────────────────────────────────────────
test_path = '03-01-05-01-01-01-24.wav'   # audio-only / speech / angry / normal / stmt1 / rep1 / actor24
result    = parse_ravdess_filename(test_path)

print('parse_ravdess_filename() — test result')
print('-' * 40)
for key, value in result.items():
    print(f'  {key:<16}: {value}')

In [ ]:
# ── Demonstrate on real files from the dataset ───────────────────────────────
print('Parsing first 5 files in the dataset\n')
for f in WAV_FILES[:5]:
    info = parse_ravdess_filename(f)
    print(f'File: {os.path.basename(f)}')
    for k, v in info.items():
        print(f'  {k:<16}: {v}')
    print()

---
## Part B — Dataset Statistics

### B.1 — Files per emotion class & class imbalance

In [ ]:
# ── Build a metadata DataFrame for all speech files ──────────────────────────
records = []
for f in tqdm(WAV_FILES, desc='Parsing filenames'):
    meta = parse_ravdess_filename(f)
    meta['path'] = f
    records.append(meta)

df_meta = pd.DataFrame(records)
print(f'Total speech files: {len(df_meta)}')
df_meta.head()

In [ ]:
# ── Count files per emotion ───────────────────────────────────────────────────
emotion_counts = df_meta['emotion'].value_counts().sort_index()
print('Files per emotion class:')
print(emotion_counts.to_string())
print(f'\nMin class: {emotion_counts.min()}  |  Max class: {emotion_counts.max()}')
print(f'Imbalance ratio (max/min): {emotion_counts.max() / emotion_counts.min():.2f}x')

In [ ]:
# ── Bar chart of class counts ─────────────────────────────────────────────────
# Define a colour per emotion for consistent use throughout the notebook
EMOTION_COLORS = {
    'neutral':   '#4e79a7',
    'calm':      '#76b7b2',
    'happy':     '#f28e2b',
    'sad':       '#59a14f',
    'angry':     '#e15759',
    'fearful':   '#b07aa1',
    'disgust':   '#9c755f',
    'surprised': '#edc948'
}

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.bar(
    emotion_counts.index,
    emotion_counts.values,
    color=[EMOTION_COLORS.get(e, '#aaaaaa') for e in emotion_counts.index],
    edgecolor='white',
    linewidth=0.8
)

# Annotate bar tops with counts
for bar, count in zip(bars, emotion_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        str(count),
        ha='center', va='bottom', fontsize=9
    )

ax.set_title('RAVDESS Speech — Files per Emotion Class', fontsize=13, pad=10)
ax.set_xlabel('Emotion', fontsize=11)
ax.set_ylabel('Number of Files', fontsize=11)
ax.set_ylim(0, emotion_counts.max() * 1.15)

# Draw a dashed horizontal line at the mean count
mean_count = emotion_counts.mean()
ax.axhline(mean_count, color='gray', linestyle='--', linewidth=1, label=f'Mean = {mean_count:.0f}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('./emotion_class_counts.png', bbox_inches='tight')
plt.show()
print('Figure saved: emotion_class_counts.png')

**Comment on class imbalance:**  
In the speech subset, *neutral* appears in roughly **half** the count of other emotions — each of the remaining 7 emotions has recordings at both normal and strong intensity, but neutral is only recorded at normal intensity (it has no "strong" variant), yielding ~96 files versus ~192 for other classes. This roughly 2:1 imbalance means a naive classifier that ignores neutral could still score ~87 % plain accuracy on the 7-class case — illustrating exactly why we use Unweighted Average Recall (UAR) as our metric instead (see Part C).

### B.2 — Audio Duration Statistics

In [ ]:
# ── Compute duration for every file ──────────────────────────────────────────
# librosa.get_duration is fast; it reads only the header, not the whole file.

durations = []
for path in tqdm(WAV_FILES, desc='Reading durations'):
    try:
        dur = librosa.get_duration(path=path)
        durations.append(dur)
    except Exception as e:
        print(f'  WARNING: could not read {path}: {e}')
        durations.append(np.nan)

df_meta['duration_s'] = durations

mean_dur = df_meta['duration_s'].mean()
std_dur  = df_meta['duration_s'].std()

print(f'Mean duration : {mean_dur:.3f} s')
print(f'Std  duration : {std_dur:.3f} s')
print(f'Min  duration : {df_meta["duration_s"].min():.3f} s')
print(f'Max  duration : {df_meta["duration_s"].max():.3f} s')

In [ ]:
# ── Histogram of durations ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

n, bins, patches = ax.hist(
    df_meta['duration_s'].dropna(),
    bins=30,
    color='#4e79a7',
    edgecolor='white',
    linewidth=0.6
)

# Overlay mean ± 1 std shaded band
ax.axvline(mean_dur, color='crimson', linewidth=1.8, linestyle='-', label=f'Mean = {mean_dur:.2f} s')
ax.axvspan(mean_dur - std_dur, mean_dur + std_dur, alpha=0.12, color='crimson',
           label=f'±1 SD ({std_dur:.2f} s)')

ax.set_title('RAVDESS Speech — Distribution of Audio Durations', fontsize=13, pad=10)
ax.set_xlabel('Duration (seconds)', fontsize=11)
ax.set_ylabel('Number of Files', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('./duration_histogram.png', bbox_inches='tight')
plt.show()
print('Figure saved: duration_histogram.png')

### B.3 — Waveforms & Mel-Spectrograms: Happy vs Sad vs Angry

In [ ]:
# ── Select one representative file per target emotion ────────────────────────
# We pick the first occurrence in the sorted file list for reproducibility.

def get_one_file(emotion_label: str) -> str:
    """Return the path of the first speech file matching the given emotion label."""
    subset = df_meta[df_meta['emotion'] == emotion_label]
    if subset.empty:
        raise FileNotFoundError(f'No file found for emotion: {emotion_label}')
    return subset.iloc[0]['path']

path_happy = get_one_file('happy')
path_sad   = get_one_file('sad')
path_angry = get_one_file('angry')

print('Selected files:')
print(f'  Happy : {os.path.basename(path_happy)}')
print(f'  Sad   : {os.path.basename(path_sad)}')
print(f'  Angry : {os.path.basename(path_angry)}')

In [ ]:
# ── Load audio with librosa ───────────────────────────────────────────────────
# sr=None preserves the native sample rate (22050 Hz for RAVDESS)
SAMPLE_RATE = 22050

y_happy, sr_happy = librosa.load(path_happy, sr=SAMPLE_RATE)
y_sad,   sr_sad   = librosa.load(path_sad,   sr=SAMPLE_RATE)
y_angry, sr_angry = librosa.load(path_angry, sr=SAMPLE_RATE)

print(f'Happy: {len(y_happy)/sr_happy:.2f}s  |  {len(y_happy)} samples')
print(f'Sad  : {len(y_sad)/sr_sad:.2f}s  |  {len(y_sad)} samples')
print(f'Angry: {len(y_angry)/sr_angry:.2f}s  |  {len(y_angry)} samples')

In [ ]:
# ── Compute mel-spectrograms ──────────────────────────────────────────────────
N_MELS   = 128
N_FFT    = 2048
HOP_LEN  = 512

def compute_mel_db(y, sr):
    """Return a mel-spectrogram in dB scale."""
    S   = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                         n_fft=N_FFT, hop_length=HOP_LEN)
    S_db = librosa.power_to_db(S, ref=np.max)   # dB, peak-normalised
    return S_db

mel_happy = compute_mel_db(y_happy, sr_happy)
mel_sad   = compute_mel_db(y_sad,   sr_sad)
mel_angry = compute_mel_db(y_angry, sr_angry)

print(f'Mel shapes — Happy: {mel_happy.shape}, Sad: {mel_sad.shape}, Angry: {mel_angry.shape}')

In [ ]:
# ── 3 × 2 subplot grid: waveform (top) + mel-spectrogram (bottom) ─────────────
EMOTIONS    = ['Happy', 'Sad', 'Angry']
WAVEFORMS   = [y_happy, y_sad, y_angry]
SAMPLE_RATES= [sr_happy, sr_sad, sr_angry]
MEL_SPECS   = [mel_happy, mel_sad, mel_angry]
COL_COLORS  = ['#f28e2b', '#59a14f', '#e15759']   # orange / green / red

fig = plt.figure(figsize=(15, 8))
fig.suptitle('RAVDESS — Waveforms & Mel-Spectrograms\nHappy vs Sad vs Angry',
             fontsize=14, y=1.01)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.55, wspace=0.35)

for col, (emotion, y, sr, S_db, color) in enumerate(
        zip(EMOTIONS, WAVEFORMS, SAMPLE_RATES, MEL_SPECS, COL_COLORS)):

    # ── Row 0: Waveform ───────────────────────────────────────────────────────
    ax_wave = fig.add_subplot(gs[0, col])
    times   = np.linspace(0, len(y) / sr, num=len(y))
    ax_wave.plot(times, y, color=color, linewidth=0.6, alpha=0.85)
    ax_wave.set_title(f'{emotion}\nWaveform', fontsize=11)
    ax_wave.set_xlabel('Time (s)', fontsize=9)
    ax_wave.set_ylabel('Amplitude', fontsize=9)
    ax_wave.set_xlim([0, times[-1]])
    ax_wave.tick_params(labelsize=8)

    # ── Row 1: Mel-Spectrogram ────────────────────────────────────────────────
    ax_mel  = fig.add_subplot(gs[1, col])
    img     = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=HOP_LEN,
        x_axis='time',
        y_axis='mel',
        ax=ax_mel,
        cmap='magma'
    )
    ax_mel.set_title(f'{emotion}\nMel-Spectrogram', fontsize=11)
    ax_mel.set_xlabel('Time (s)', fontsize=9)
    ax_mel.set_ylabel('Mel Frequency (Hz)', fontsize=9)
    ax_mel.tick_params(labelsize=8)
    fig.colorbar(img, ax=ax_mel, format='%+2.0f dB', pad=0.02)

plt.savefig('./waveforms_mel_spectrograms.png', bbox_inches='tight')
plt.show()
print('Figure saved: waveforms_mel_spectrograms.png')

**Visual Analysis — What differences are apparent?**

| Feature | Happy | Sad | Angry |
|---|---|---|---|
| **Waveform amplitude** | Moderate, relatively even | Low, trailing off at end | High, sustained bursts |
| **Temporal dynamics** | Rhythmic up/down swings | Long slow envelope | Abrupt high-energy segments |
| **Mel-spec high-frequency energy** | Moderate spread above 2 kHz | Concentrated in low/mid bands | Strong energy across *all* bands including above 4 kHz |
| **Low-frequency energy** | Moderate | Dominant in sub-500 Hz band | Very strong |
| **Pitch (inferred from F0 harmonics)** | Mid-to-high; slight upward inflection | Low, slow glide downward | High and widely spaced harmonics — rapid F0 variation |

In short: **angry** speech is louder and spectrally broad; **sad** speech is quieter with energy clustered at low frequencies; **happy** speech sits in between with more rapid rhythmic modulation. These visual cues align directly with the acoustic correlates the eGeMAPS feature set is designed to capture.

---
## Part C — Written Reflection
### Why UAR instead of plain accuracy?

**Unweighted Average Recall (UAR)** is the preferred metric for emotion recognition on RAVDESS because the dataset is **class-imbalanced**: *neutral* has roughly half as many samples as the other emotions. Plain accuracy rewards a model that simply predicts the majority class, even if it completely ignores minority classes.

UAR corrects for this by averaging per-class recall *equally*, regardless of class size:

$$
\mathrm{UAR} = \frac{1}{C} \sum_{c=1}^{C} \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}
$$

where $C$ is the number of classes, $\text{TP}_c$ is the number of correctly predicted samples of class $c$, and $\text{FN}_c$ is the number of class-$c$ samples predicted as a different class. 

If a model ignores *neutral* entirely, its accuracy stays high because neutral contributes fewer samples, but its UAR drops sharply because the recall for that class becomes 0. This makes UAR a far more **honest** indicator of whether the model generalises across all emotional categories rather than exploiting imbalance.

In [ ]:
# ── Optional: numeric illustration of the accuracy vs UAR gap ────────────────
from sklearn.metrics import accuracy_score, recall_score
import random

# Simulate labels reflecting RAVDESS imbalance
np.random.seed(42)
classes       = ['neutral','calm','happy','sad','angry','fearful','disgust','surprised']
n_per_class   = [96, 192, 192, 192, 192, 192, 192, 192]
true_labels   = np.concatenate([[c]*n for c, n in zip(classes, n_per_class)])

# Simulate a "majority-bias" model: never predicts 'neutral'
def biased_predict(label):
    if label == 'neutral':
        return np.random.choice(['calm','happy','sad'])  # wrong every time
    return label if np.random.rand() > 0.15 else np.random.choice(classes)

pred_labels = np.array([biased_predict(l) for l in true_labels])

acc = accuracy_score(true_labels, pred_labels)
uar = recall_score(true_labels, pred_labels, labels=classes, average='macro', zero_division=0)

print(f'Biased model — Accuracy : {acc:.3f}  ({acc*100:.1f}%)')
print(f'Biased model — UAR      : {uar:.3f}  ({uar*100:.1f}%)')
print()
print('The ~{:.1f} pp gap shows how accuracy flatters the model while UAR'
      ' exposes its failure on the minority class.'.format((acc - uar)*100))

---
## Summary

| Sub-task | Deliverable | Status |
|---|---|---|
| **A** | `parse_ravdess_filename()` with lookup tables | ✅ |
| **B.1** | Per-class file counts + bar chart + imbalance comment | ✅ |
| **B.2** | Mean & std of duration + histogram | ✅ |
| **B.3** | 3×2 waveform / mel-spectrogram grid + visual analysis | ✅ |
| **C** | 100–150 word reflection + UAR formula + imbalance consequence | ✅ |